# ASL: Inspect Correct and Incorrect Landmark Sequences

Use your Kaggle remote Python kernel. This self-contained notebook **does not train or load a model and does not import TensorFlow**. It reads a saved `validation_predictions.csv` and the original landmark parquet files.

By default, inspect the body-reference run for split 44 and training seed 42 from the latest available notebook 06 study. Set `RUN_DIR` explicitly to inspect another run, or change `ARTIFACT_ROOT` if you restored artifacts through a Kaggle dataset. The chosen run and prediction counts are displayed before inspection.

Focus on `mouth`, `tongue`, and `awake`; for awake errors, prefer `awake -> wake` when available. Randomly select at most two correct and two incorrect examples per class. The gallery displays one correct/error pair per class, with a placeholder when one group is empty.

Animations show **landmarks, not original video**. Playback is uniformly resampled for display, not original timing. A missing landmark or an unusual motion is an observation, not proof of a wrong label, cropped recording, or signer identity. Do not use this notebook to identify people.


In [ ]:
print('Starting landmark error inspection; TensorFlow is not required.', flush=True)
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import display, HTML

ARTIFACT_ROOT = Path('/kaggle/working/asl-project')
RUN_DIR = None  # Or Path('/kaggle/working/asl-project/<study>/<split>/<run>').
PREFERRED_SPLIT_SEED = 44
PREFERRED_TRAIN_SEED = 42
DATA_DIR = None
INPUT_DIR = Path('/kaggle/input')
FOCUS_SIGNS = ['mouth', 'tongue', 'awake']
TARGET_CONFUSIONS = {'awake': 'wake'}
SELECTION_SEED = 42
EXAMPLES_PER_GROUP = 2
MAX_ANIMATION_FRAMES = 40
DISPLAY_FPS = 10
QUALITY_SAMPLE_LIMIT = 500  # Balanced by correctness, not a population estimate.

if not INPUT_DIR.is_dir():
    raise RuntimeError('Select a Kaggle remote kernel; /kaggle/input is unavailable.')
if EXAMPLES_PER_GROUP < 1 or MAX_ANIMATION_FRAMES < 2 or DISPLAY_FPS <= 0:
    raise ValueError('Use positive gallery settings and at least two animation frames.')
if QUALITY_SAMPLE_LIMIT < 2:
    raise ValueError('QUALITY_SAMPLE_LIMIT must be at least 2.')


In [ ]:
if RUN_DIR is None:
    pattern = (f'signer_splits_*/split_{PREFERRED_SPLIT_SEED}/'
               f'body_reference_seed_{PREFERRED_TRAIN_SEED}/validation_predictions.csv')
    candidates = sorted(ARTIFACT_ROOT.glob(pattern))
    if not candidates:
        raise FileNotFoundError(
            f'No matching saved run found under {ARTIFACT_ROOT}. Set RUN_DIR explicitly '
            'or restore predictions and split metadata from a saved Kaggle output.')
    RUN_DIR = candidates[-1].parent
RUN_DIR = Path(RUN_DIR)
prediction_file = RUN_DIR/'validation_predictions.csv'
if not prediction_file.is_file():
    raise FileNotFoundError(f'Missing {prediction_file}; no retraining is necessary if saved predictions exist.')
predictions = pd.read_csv(prediction_file)
required = {'path', 'participant_id', 'sign', 'predicted_sign', 'confidence'}
if not required.issubset(predictions.columns):
    raise ValueError(f'Predictions are missing columns: {required-set(predictions.columns)}')
if predictions.empty or predictions['path'].duplicated().any():
    raise ValueError('Predictions must be nonempty with unique sample paths.')
if predictions[list(required)].isna().any().any():
    raise ValueError('Prediction metadata contains missing values.')
# Match predictions to saved validation metadata whenever it is available.
validation_file = None
for folder in [RUN_DIR, RUN_DIR.parent, RUN_DIR.parent.parent]:
    if (folder/'validation_split.csv').is_file():
        validation_file = folder/'validation_split.csv'
        break
if validation_file is None:
    raise FileNotFoundError('Restore validation_split.csv with the run to verify sample alignment.')
validation = pd.read_csv(validation_file)
columns = ['path','participant_id','sign']
if not predictions[columns].astype(str).equals(validation[columns].astype(str)):
    raise ValueError('Predictions are not aligned with the saved validation partition.')
predictions['correct'] = predictions['sign'].eq(predictions['predicted_sign'])
OUTPUT_DIR = Path('/kaggle/working/asl-project') / ('error_inspection_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print('Selected run:', RUN_DIR)
print('Validation metadata:', validation_file)
print('Samples:', len(predictions), '| accuracy:', predictions['correct'].mean())
print('Inspection outputs:', OUTPUT_DIR)
(OUTPUT_DIR/'inspection_config.json').write_text(json.dumps(dict(
    run_dir=str(RUN_DIR), validation_file=str(validation_file), focus_signs=FOCUS_SIGNS,
    selection_seed=SELECTION_SEED, examples_per_group=EXAMPLES_PER_GROUP,
    max_animation_frames=MAX_ANIMATION_FRAMES, display_fps=DISPLAY_FPS), indent=2))


In [ ]:
# To specify the dataset manually, use Path('/kaggle/input/competitions/asl-signs').
# Search only a few directory levels to avoid traversing every parquet file.
if DATA_DIR is None:
    candidates = []
    for pattern in ('train.csv', '*/train.csv', '*/*/train.csv', '*/*/*/train.csv'):
        for csv_path in INPUT_DIR.glob(pattern):
            if (csv_path.parent / 'sign_to_prediction_index_map.json').is_file():
                candidates.append(csv_path.parent)
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            'ASL competition data was not found. In your Kaggle notebook, use Add Input to add '
            'Google - Isolated Sign Language Recognition (asl-signs), then rerun this cell.'
        )
    if len(candidates) > 1:
        raise RuntimeError(f'Multiple dataset directories found; set DATA_DIR explicitly above: {candidates}')
    DATA_DIR = candidates[0]
print('Data directory:', DATA_DIR)

table = pd.read_csv(DATA_DIR / 'train.csv')
labels = json.loads((DATA_DIR / 'sign_to_prediction_index_map.json').read_text())
required = {'path', 'sign', 'participant_id'}
if not required.issubset(table.columns):
    raise ValueError(f'train.csv is missing required columns: {required - set(table.columns)}')
if table.empty:
    raise ValueError('train.csv contains no samples')
if sorted(labels.values()) != list(range(len(labels))):
    raise ValueError('Label indices must be contiguous and start at zero')
if not table['sign'].isin(labels).all():
    raise ValueError('train.csv contains labels missing from the official label mapping')
print('Samples:', len(table))
print('Classes:', len(labels))
print('Signers:', table['participant_id'].nunique())
display(table.head())
display(table.groupby('participant_id').size().rename('samples').to_frame())

if not predictions['sign'].isin(labels).all() or not predictions['predicted_sign'].isin(labels).all():
    raise ValueError('Prediction labels do not match this dataset label map.')
lookup = table[['path','participant_id','sign']].astype(str).set_index('path')
for row in predictions.itertuples(index=False):
    if row.path not in lookup.index:
        raise ValueError(f'Prediction sample is missing from the attached dataset: {row.path}')
    meta = lookup.loc[row.path]
    if str(row.participant_id) != meta['participant_id'] or row.sign != meta['sign']:
        raise ValueError(f'Prediction metadata differs from the attached dataset: {row.path}')


In [ ]:
class_summary = predictions.groupby('sign').agg(support=('correct','size'),
    correct=('correct','sum'), recall=('correct','mean'))
class_summary.to_csv(OUTPUT_DIR/'class_recall.csv')
display(class_summary.reindex([s for s in FOCUS_SIGNS if s in class_summary.index]))
wrong = predictions[~predictions['correct']]
confusions = wrong.groupby(['sign','predicted_sign']).size().rename('count').reset_index()
confusions['class_support'] = confusions['sign'].map(class_summary['support'])
confusions['class_error_fraction'] = confusions['count']/confusions['class_support']
confusions = confusions.sort_values('count',ascending=False)
confusions.to_csv(OUTPUT_DIR/'confusion_pairs.csv',index=False)
display(confusions.head(20))
signer_summary = predictions.groupby('participant_id').agg(
    samples=('correct','size'), accuracy=('correct','mean'))
signer_summary.to_csv(OUTPUT_DIR/'validation_signer_accuracy.csv')
display(signer_summary)

selected = []
for sign in FOCUS_SIGNS:
    for correct in (True,False):
        pool = predictions[(predictions['sign']==sign) & (predictions['correct']==correct)]
        selection_rule = 'random_within_group'
        if not correct and sign in TARGET_CONFUSIONS:
            targeted = pool[pool['predicted_sign']==TARGET_CONFUSIONS[sign]]
            if not targeted.empty:
                pool = targeted
                selection_rule = 'targeted_confusion'
        if pool.empty:
            print('No examples:', sign, '|', 'correct' if correct else 'incorrect')
            continue
        sample = pool.sample(n=min(EXAMPLES_PER_GROUP,len(pool)),random_state=SELECTION_SEED).copy()
        sample['selection_rule'] = selection_rule
        selected.append(sample)
selected = (pd.concat(selected,ignore_index=True) if selected else
            pd.DataFrame(columns=list(predictions.columns)+['selection_rule']))
selected.to_csv(OUTPUT_DIR/'selected_examples.csv',index=False)
display(selected)


In [ ]:
def load_frames(path):
    import numpy as np
    import pandas as pd
    data = pd.read_parquet(path, columns=["x", "y", "z"]).to_numpy(dtype=np.float32)
    if len(data) == 0 or len(data) % 543:
        raise ValueError(f"Invalid landmark rows in {path}: {len(data)}")
    return data.reshape(-1, 543, 3)

def clip_reference(frames, hands, valid):
    """Return a stable origin, positive scalar scale, and reference source."""
    xy = np.asarray(frames[..., :2], dtype=np.float64)
    shoulders = xy[:, [489 + 11, 489 + 12], :]
    shoulder_valid = np.isfinite(shoulders).all(axis=(1, 2))
    pairs = shoulders[shoulder_valid]
    if len(pairs):
        widths = np.linalg.norm(pairs[:, 0] - pairs[:, 1], axis=1)
        usable = np.isfinite(widths) & (widths > 1e-6)
        if usable.any():
            center = np.median(pairs[usable].mean(axis=1), axis=0)
            scale = float(np.median(widths[usable]))
            return center, max(scale, 1e-6), 'shoulders'

    face = xy[:, :468, :]
    centers, spreads = [], []
    for points in face:
        points = points[np.isfinite(points).all(axis=1)]
        if len(points) < 3:
            continue
        center = np.median(points, axis=0)
        spread = float(np.sqrt(np.mean(np.sum((points - center) ** 2, axis=1))))
        if np.isfinite(spread) and spread > 1e-6:
            centers.append(center)
            spreads.append(spread)
    if centers:
        return np.median(centers, axis=0), float(np.median(spreads)), 'face'

    points = np.asarray(hands[valid], dtype=np.float64)
    if len(points):
        center = np.median(points, axis=0)
        spread = float(np.sqrt(np.mean(np.sum((points - center) ** 2, axis=1))))
        return center, max(spread, 1e-6), 'hands_fallback'
    return np.zeros(2), 1.0, 'no_hands'




def longest_gap(mask):
    longest = current = 0
    for missing in mask:
        current = current + 1 if missing else 0
        longest = max(longest,current)
    return int(longest)


def inspect_quality(path):
    frames = load_frames(path)
    hands = frames[:, list(range(468,489))+list(range(522,543)), :2]
    valid = np.isfinite(hands).all(axis=-1)
    left, right = valid[:,:21].any(axis=1), valid[:,21:].any(axis=1)
    shoulders = frames[:,[500,501],:2]
    shoulder_ok = np.isfinite(shoulders).all(axis=(1,2))
    center, scale, reference = clip_reference(frames,hands,valid)
    del center
    wrist_steps = []
    for index in [468,522]:
        wrist = frames[:,index,:2]
        pair_valid = np.isfinite(wrist[:-1]).all(axis=1) & np.isfinite(wrist[1:]).all(axis=1)
        distances = np.linalg.norm(np.diff(wrist,axis=0)[pair_valid],axis=1)/scale
        wrist_steps.extend(distances.tolist())
    return dict(frames=len(frames), missing_hand_xy_fraction=float(1-valid.mean()),
        no_hand_frame_fraction=float((~(left|right)).mean()),
        no_left_hand_fraction=float((~left).mean()), no_right_hand_fraction=float((~right).mean()),
        longest_left_gap=longest_gap(~left), longest_right_gap=longest_gap(~right),
        shoulder_pair_fraction=float(shoulder_ok.mean()), reference_source=reference,
        median_wrist_step_body_units=(float(np.median(wrist_steps)) if wrist_steps else np.nan))

# Balance correctness groups for exploration; this is intentionally not a random population sample.
quality_rows = []
for correct in (True,False):
    pool = predictions[predictions['correct']==correct]
    quality_rows.append(pool.sample(n=min(QUALITY_SAMPLE_LIMIT//2,len(pool)),random_state=SELECTION_SEED))
quality_sample = pd.concat(quality_rows,ignore_index=True)
quality_sample = pd.concat([quality_sample,selected[list(predictions.columns)]],ignore_index=True).drop_duplicates('path')
quality_records = []
for i,row in enumerate(quality_sample.itertuples(index=False)):
    quality_records.append(dict(path=row.path,sign=row.sign,correct=row.correct,
                                **inspect_quality(DATA_DIR/row.path)))
    if (i+1)%100==0 or i+1==len(quality_sample):
        print(f'Quality inspection: {i+1}/{len(quality_sample)}',flush=True)
quality = pd.DataFrame(quality_records)
quality.to_csv(OUTPUT_DIR/'landmark_quality_sample.csv',index=False)
quality_summary = quality.groupby('correct').agg(
    inspected=('path','size'), median_frames=('frames','median'),
    median_missing_hand_xy_fraction=('missing_hand_xy_fraction','median'),
    median_no_hand_frame_fraction=('no_hand_frame_fraction','median'),
    median_shoulder_pair_fraction=('shoulder_pair_fraction','median'))
quality_summary.to_csv(OUTPUT_DIR/'quality_summary.csv')
display(quality_summary)
display(quality[quality['path'].isin(selected['path'])])
print('Quality summaries use a selected, correctness-balanced subset; they are not causal explanations or unbiased dataset estimates.')


## Landmark animations

The two panels use the same body-referenced coordinate scale and fixed axis limits for the selected pair. Face landmarks are shown sparsely in gray for context; colored hand points and lines show hand structure. Shoulder/elbow/nose pose points are black. Both panels use display-resampled frame indices, so side-by-side playback is not a comparison of original signing speed.

Coordinates are displayed as recorded (no left/right mirroring). Missing points disappear rather than being interpolated. Axis bounds use percentiles to limit extreme tracking outliers, so some outliers may lie outside the view. Confidence is a model output, not a verified probability of correctness.


In [ ]:
HAND_EDGES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
              (5,9),(9,10),(10,11),(11,12),(9,13),(13,14),(14,15),(15,16),
              (13,17),(0,17),(17,18),(18,19),(19,20)]
POSE_INDICES = [489,500,501,502,503]


def animation_data(row):
    if row is None:
        return None
    frames = load_frames(DATA_DIR/row['path'])
    hands = frames[:,list(range(468,489))+list(range(522,543)),:2]
    valid = np.isfinite(hands).all(axis=-1)
    center,scale,source = clip_reference(frames,hands,valid)
    xy = (frames[...,:2]-center)/scale
    xy[~np.isfinite(xy).all(axis=-1)] = np.nan
    return dict(xy=xy,row=row,source=source)


def animate_pair(sign,correct_row,error_row):
    clips = [animation_data(correct_row),animation_data(error_row)]
    present = [clip for clip in clips if clip is not None]
    if not present:
        return None
    plotted = list(range(0,468,8))+list(range(468,489))+POSE_INDICES+list(range(522,543))
    points = np.concatenate([c['xy'][:,plotted,:].reshape(-1,2) for c in present])
    points = points[np.isfinite(points).all(axis=1)]
    if len(points):
        low,high = np.percentile(points,[1,99],axis=0)
        span = max(float(np.max(high-low)),0.5)
        midpoint = (low+high)/2
    else:
        span,midpoint = 1.0,np.zeros(2)
    limits = (midpoint-span*0.65,midpoint+span*0.65)
    count = min(MAX_ANIMATION_FRAMES,max(len(c['xy']) for c in present))
    indices = [np.rint(np.linspace(0,len(c['xy'])-1,count)).astype(int) if c else None for c in clips]
    fig,axes = plt.subplots(1,2,figsize=(10,4))
    def draw(frame_number):
        for panel,(axis,clip) in enumerate(zip(axes,clips)):
            axis.clear()
            axis.set_xlim(limits[0][0],limits[1][0])
            axis.set_ylim(limits[1][1],limits[0][1])
            axis.set_aspect('equal',adjustable='box')
            axis.set_xlabel('Body-reference x')
            if clip is None:
                axis.set_title('No correct example available' if panel==0 else 'No error example available')
                continue
            raw_frame = int(indices[panel][frame_number])
            xy = clip['xy'][raw_frame]
            face = xy[:468:8]
            axis.scatter(face[:,0],face[:,1],s=3,c='gray',alpha=0.4)
            pose = xy[POSE_INDICES]
            axis.scatter(pose[:,0],pose[:,1],s=12,c='black')
            axis.plot(xy[[500,501],0],xy[[500,501],1],c='black',lw=1)
            for start,color in [(468,'tab:blue'),(522,'tab:orange')]:
                hand = xy[start:start+21]
                axis.scatter(hand[:,0],hand[:,1],s=9,c=color)
                for a,b in HAND_EDGES:
                    axis.plot(hand[[a,b],0],hand[[a,b],1],c=color,lw=1)
            row = clip['row']
            status = 'Correct' if panel==0 else 'Incorrect'
            axis.set_title(f"{status}: {row['sign']} -> {row['predicted_sign']}\n"
                           f"confidence={row['confidence']:.2f} | source frame {raw_frame}")
        fig.suptitle(f'{sign}: landmark inspection (display timing only)')
        fig.tight_layout()
    animation = FuncAnimation(fig,draw,frames=count,interval=1000/DISPLAY_FPS)
    html = animation.to_jshtml(fps=DISPLAY_FPS)
    plt.close(fig)
    safe_name = ''.join(ch if ch.isalnum() or ch in '-_' else '_' for ch in sign)
    (OUTPUT_DIR/f'{safe_name}_correct_vs_error.html').write_text(html,encoding='utf-8')
    display(HTML(html))
    return html

for sign in FOCUS_SIGNS:
    group = selected[selected['sign']==sign]
    correct = group[group['correct']]
    wrong = group[~group['correct']]
    animate_pair(sign,correct.iloc[0] if len(correct) else None,
                 wrong.iloc[0] if len(wrong) else None)


In [ ]:
# Fill this review sheet after watching the animations. Do not infer a label error
# solely because the classifier was wrong or one hand was missing.
review = selected[['path','participant_id','sign','predicted_sign','correct','selection_rule']].copy()
review['tracking_observations'] = ''
review['motion_observations'] = ''
review['hypothesis_to_test'] = ''
review['review_status'] = 'pending_manual_review'
review.to_csv(OUTPUT_DIR/'manual_review_template.csv',index=False)
print('Saved reports, animations, and manual review template to:',OUTPUT_DIR)


## What to inspect next

For a single sign, compare correct and incorrect examples: visible hands, long tracking gaps, reference stability, hand shape, and movement relative to the face/body. A lack of one hand can be a valid one-handed sign. A tracking gap does not prove that the original video was cropped. Different movement patterns can be valid sign variants.

Record observations before choosing a change. If tracking gaps dominate, investigate missing-data handling; if position is preserved but movement order appears important, design a temporal-feature or temporal-model experiment. Do not claim either cause from aggregate accuracy alone.

This gallery is selected development data. Avoid treating its quality summaries as population error rates or repeatedly tuning against it as though it were an untouched final test. Save the review files before the Kaggle session ends. Do not commit raw landmarks or session URLs.
